# 10.1 Das Euler-Verfahren

In Kapitel 7 haben wir aus bekannten Positionswerten Ableitungen berechnet,
in Kapitel 9 aus bekannten Funktionswerten Integrale. Jetzt stellen wir die
Frage anders: Ein Metallstab mit Anfangstemperatur $T_0 = 80\,°C$ liegt in
einem Raum mit Umgebungstemperatur $T_\infty = 20\,°C$. Wir wissen, dass die
Temperatur schneller fällt, wenn der Stab noch heiß ist, und immer langsamer,
je näher er der Umgebungstemperatur kommt. Das **Newtonsche Abkühlungsgesetz**
fasst das in einer einzigen Gleichung zusammen:

$$\dot{T}(t) = -k\,\bigl(T(t) - T_\infty\bigr).$$

Wir kennen also nicht $T(t)$ direkt, sondern nur die Formel für die
Änderungsrate. Eine Gleichung, die eine unbekannte Funktion und ihre Ableitung
verknüpft, heißt **gewöhnliche Differentialgleichung** (kurz DGL). Gesucht ist
der Temperaturverlauf $T(t)$ für $t \geq 0$.

## Lernziele

* [ ] Sie können erklären, was eine **gewöhnliche Differentialgleichung 1.
  Ordnung** ist, und ein physikalisches Beispiel nennen.
* [ ] Sie können das **explizite Euler-Verfahren** aus dem
  Vorwärts-Differenzenquotienten herleiten und als Python-Schleife
  implementieren.
* [ ] Sie wissen, was **Schrittweite** und **Anfangsbedingung** bedeuten, und
  können beschreiben, wie die Schrittweite die Genauigkeit beeinflusst.
* [ ] Sie können die Euler-Lösung mit einer analytischen Referenzlösung
  vergleichen und den globalen Fehler quantifizieren.

## Wie sieht die Abkühlung aus?

Bevor wir das Abkühlproblem numerisch lösen, verschaffen wir uns einen
Überblick: Wie sieht die Lösung eigentlich aus? Für das Newtonsche
Abkühlungsgesetz lässt sich die Gleichung analytisch lösen. Wir trennen die
Variablen

\begin{equation*}
\frac{dT}{dt}=-k\,(T-T_{\infty}) \quad\Rightarrow\quad
\frac{dT}{T-T_{\infty}}=-k\, dt,
\end{equation*}

integrieren auf beiden Seiten

\begin{equation*}
\ln |T - T_{\infty}| = -kt + \tilde{C}
\end{equation*}

und erhalten die allgemeine Lösung

\begin{equation*}
T(t) = T_{\infty} + C\, e^{-kt}.
\end{equation*}

Mit der Anfangsbedingung $T(0)=T_0$ lautet die Lösung

\begin{equation*}
T(t) = T_{\infty} + (T_0 - T_{\infty}) e^{-kt}.
\end{equation*}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

# --- Physikalisches Modell: Newtonsches Abkühlungsgesetz ---
# dT/dt = -k * (T - T_inf)
# Anfangstemperatur T0, Umgebungstemperatur T_inf, Abkühlkonstante k

T0    = 80.0   # Anfangstemperatur in °C
T_inf = 20.0   # Umgebungstemperatur in °C
k     = 0.1    # Abkühlkonstante in 1/min

# --- Analytische Lösung (Referenz für alle weiteren Abschnitte) ---
# Durch Trennung der Variablen ergibt sich: T(t) = T_inf + (T0 - T_inf) * exp(-k*t)
# Diese Kurve ist unsere "Wahrheit", gegen die wir die numerischen Methoden messen.
t_fein  = np.linspace(0, 50, 500)
T_exakt = T_inf + (T0 - T_inf) * np.exp(-k * t_fein)

# --- Ausgabe ---
print(f"Anfangstemperatur:        {T0:.1f} °C")
print(f"Umgebungstemperatur:      {T_inf:.1f} °C")
print(f"Temperatur nach 10 min:   {T_inf + (T0 - T_inf) * np.exp(-k * 10):.2f} °C")
print(f"Temperatur nach 30 min:   {T_inf + (T0 - T_inf) * np.exp(-k * 30):.2f} °C")
print(f"Temperatur nach 50 min:   {T_inf + (T0 - T_inf) * np.exp(-k * 50):.2f} °C")

# --- Visualisierung ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_fein, T_exakt, label='analytische Lösung')
ax.axhline(T_inf, color='black', linestyle='--',
           label=f'Gleichgewicht $T_\\infty = {T_inf:.0f}\\,°C$')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Newtonsches Abkühlungsgesetz: analytische Lösung')
ax.legend()
ax.grid(True)
plt.show()

Die Temperatur nähert sich asymptotisch der Umgebungstemperatur $T_{\infty} =
20~^{\circ}\text{C}$. Nach 50 min ist der Unterschied bereits kleiner als 1 °C.
Die Funktion $\dot{T}(t) = -k\,(T - T_\infty)$ beschreibt eine einfache, aber
wichtige Klasse von DGLen: **autonome Differentialgleichungen 1. Ordnung**.
"Autonom" bedeutet, dass die rechte Seite nicht explizit von der Zeit $t$
abhängt, sondern nur vom aktuellen Zustand $T$.

| Begriff | Bedeutung im Beispiel |
| --- | --- |
| **DGL 1. Ordnung** | Nur die erste Ableitung $\dot{T}$ taucht auf, keine $\ddot{T}$ |
| **Anfangsbedingung** | $T(0) = T_0 = 80~^{\circ}\text{C}$; legt die spezifische Lösungskurve fest |
| **rechte Seite** $f(T)$ | $-k\,(T - T_\infty)$; gibt die Änderungsrate als Funktion des Zustands an |
| **autonome DGL** | $f$ hängt nicht explizit von $t$ ab |

*Was passiert, wenn die Heizung im Raum läuft und $T_\infty$ selbst zeitabhängig
ist?* Dann ist die rechte Seite $f(t, T)$ und die DGL heißt nicht-autonom.
Diesen Fall behandeln wir in Abschnitt 10.3.

### Mini-Übung 1

1. Die analytische Lösung lautet $T(t) = T_\infty + (T_0 - T_\infty)\,e^{-kt}$.
   Was ergibt sich für $T(0)$? Überprüfen Sie, dass die Anfangsbedingung
   erfüllt ist.
2. Für welchen Wert von $T$ gilt $\dot{T} = 0$? Was bedeutet das physikalisch,
   und warum ist das im Plot erkennbar?
3. Was würde sich ändern, wenn $k$ verdoppelt wird: Ändert sich die
   Gleichgewichtstemperatur? Ändert sich die Geschwindigkeit der Abkühlung?
   Beantworten Sie die Frage ohne Code.

## Ein Schritt nach vorne: das Euler-Verfahren

Wenn wir keine analytische Lösung kennen oder die Gleichung zu kompliziert
für eine geschlossene Formel ist, müssen wir die Lösung schrittweise
berechnen. Die Idee ist dieselbe wie in Kapitel 7 beim
Vorwärts-Differenzenquotienten. Wir schreiben:

$$\dot{T}(t_i) \approx \frac{T(t_{i+1}) - T(t_i)}{\Delta t}.$$

Die DGL sagt uns, dass $\dot{T}(t_i) = f(T_i)$. Setzen wir das ein und
lösen nach $T_{i+1}$ auf:

$$T_{i+1} = T_i + \Delta t \cdot f(T_i).$$

Das ist das **explizite Euler-Verfahren**: vom bekannten Zustand $T_i$ einen
Schritt der Länge $\Delta t$ in Richtung der aktuellen Steigung gehen.

In [ ]:
# --- Parameter des Euler-Verfahrens ---
t_end = 50.0   # Ende der Simulationsdauer in min
dt    = 5.0    # Schrittweite in min (bewusst grob für Visualisierung)

# --- Rechte Seite der DGL als Python-Funktion ---
def f_abkuehlung(T):
    # Änderungsrate: dT/dt = -k * (T - T_inf)
    return -k * (T - T_inf)

# --- Euler-Schleife ---
n_schritte = int(t_end / dt)            # Anzahl der Zeitschritte
t_euler    = np.zeros(n_schritte + 1)   # Zeitpunkte (inklusive t=0)
T_euler    = np.zeros(n_schritte + 1)   # Temperatur (inklusive T0)

# Anfangsbedingung
t_euler[0] = 0.0
T_euler[0] = T0

for i in range(n_schritte):
    # Euler-Update: T[i+1] = T[i] + dt * f(T[i])
    t_euler[i + 1] = t_euler[i] + dt
    T_euler[i + 1] = T_euler[i] + dt * f_abkuehlung(T_euler[i])

# --- Analytische Referenz an denselben Zeitpunkten zum Vergleich ---
T_exakt_euler = T_inf + (T0 - T_inf) * np.exp(-k * t_euler)

# --- Ausgabe der ersten Schritte ---
print(f"{'t [min]'} \t {'Euler [°C]'} \t {'Exakt [°C]'} \t {'Fehler [°C]'}")
print("-" * 52)
for i in range(n_schritte + 1):
    print(f"{t_euler[i]:.1f} \t {T_euler[i]:.4f} \t "
          f"{T_exakt_euler[i]:.4f} \t {T_euler[i] - T_exakt_euler[i]:.4f}")

# --- Plot: Euler vs. analytisch ---
t_fein  = np.linspace(0, t_end, 500)
T_fein  = T_inf + (T0 - T_inf) * np.exp(-k * t_fein)

# --- Visualisierung ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_fein, T_fein, label='analytische Lösung')
ax.plot(t_euler, T_euler, linestyle='--',
        marker='o', markersize=6, label=f'Euler-Verfahren')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Euler-Verfahren vs. analytische Lösung')
ax.legend()
ax.grid(True)
plt.show()

Mit $\Delta t = 5$ min ist der Fehler bereits mit bloßem Auge sichtbar. Das
Euler-Verfahren unterschätzt in jedem Schritt die Temperatur leicht, weil es die
Steigung am linken Rand des Intervalls als konstant annimmt. In Wirklichkeit
nimmt die Abkühlrate während des Intervalls aber kontinuierlich ab, so dass der
Euler-Schritt die Temperatur zu stark absenkt.

### Mini-Übung 2

1. Verfolgen Sie den ersten Euler-Schritt von Hand.
   Die Formel lautet: $T_1 = T_0 + \Delta t \cdot f(T_0)$.
   Setzen Sie $T_0 = 80~^{\circ}\text{C}$, $\Delta t = 5$ und $f(T_0) = -k\,(T_0 - T_\infty)$
   ein. Welchen Wert erhalten Sie für $T_1$? Lesen Sie den Wert anschließend
   aus der Tabelle ab und prüfen Sie Ihre Rechnung.
2. Ist der Fehler nach dem ersten Schritt positiv oder negativ? Was bedeutet
   das physikalisch: Überschätzt oder unterschätzt das Euler-Verfahren die
   Temperatur?

In [ ]:
# Code-Zelle

## Schrittweite und Genauigkeit

*Wie genau muss die Schrittweite sein?* Das hängt vom Problem ab, aber ein
einfaches Experiment zeigt den Zusammenhang sehr deutlich. Dazu lagern wir
zuerst das Euler-Verfahren in eine eigene Funktion aus.

In [ ]:
# --- Parameter ---
T0    = 80.0
T_inf = 20.0
k     = 0.1
t_end = 50.0

def euler_abkuehlung(T0, T_inf, k, t_end, dt):
    """Euler-Verfahren für das Newtonsche Abkühlungsgesetz.

    Parameters
    ----------
    T0    : float  Anfangstemperatur in °C
    T_inf : float  Umgebungstemperatur in °C
    k     : float  Abkühlkonstante in 1/min
    t_end : float  Simulationsende in min
    dt    : float  Schrittweite in min

    Returns
    -------
    t, T  : np.ndarray  Zeitpunkte und Temperaturwerte
    """
    n_schritte = int(t_end / dt)
    t = np.zeros(n_schritte + 1)
    T = np.zeros(n_schritte + 1)
    t[0] = 0.0
    T[0] = T0
    for i in range(n_schritte):
        # Euler-Update: T[i+1] = T[i] + dt * f(T[i])
        t[i + 1] = t[i] + dt
        T[i + 1] = T[i] + dt * (-k * (T[i] - T_inf))
    return t, T

Danach vergleichen wir die analytische Lösung mit dem Euler-Verfahren zu drei
verschiedenen Schrittweiten.

In [ ]:
# --- analytische Lösung ---
t_fein  = np.linspace(0, t_end, 500)
T_fein  = T_inf + (T0 - T_inf) * np.exp(-k * t_fein)

# --- Drei Schrittweiten im Vergleich ---
schrittweiten = [10.0, 2.0, 0.5]

# --- Euler-Verfahren und Plot ---
fig, ax = plt.subplots()
ax.plot(t_fein, T_fein, label='analytisch (Referenz)')
for dt in schrittweiten:
    t, T = euler_abkuehlung(T0, T_inf, k, t_end, dt)
    ax.plot(t, T, marker='o', markersize=4, label=f'Delta t = {dt} min')
ax.set_xlabel('Zeit in min')
ax.set_ylabel('Temperatur in °C')
ax.set_title('Temperaturverlauf')
ax.legend()
ax.grid(True)

plt.show()

Im Plot sehen wir, dass kleinere Schrittweiten die Euler-Kurve immer näher an
die analytische Lösung heranbringen. Um das genauer zu quantifizieren,
betrachten wir nun den Unterschied zwischen numerischer und analytischer Lösung
zu einem festen Zeitpunkt, z.B. bei $t = 20~\text{min}$. Als **lokaler Fehler**
bezeichnen wir den Fehler eines einzelnen Euler-Schritts, wenn man annimmt, dass
der Startwert dieses Schritts exakt ist; der Abstand zwischen numerischer und
analytischer Lösung nach vielen Schritten heißt **globaler Fehler**. Der im
nächsten Plot gezeigte Fehler bei $t = 20~\text{min}$ ist also ein globaler
Fehler.

In [ ]:
# --- Berechnung absoluter Fehler bei t = 20 min als Funktion der Schrittweite ---
schrittweiten = [10.0, 5.0, 2.0, 1.0, 0.5, 0.2, 0.1]
fehler_bei_20 = []
T_exakt_20    = T_inf + (T0 - T_inf) * np.exp(-k * 20)

for dt in schrittweiten:
    t, T = euler_abkuehlung(T0, T_inf, k, t_end, dt)
    # Gitterpunkt zu t = 20 min
    idx = int(20.0 / dt)
    fehler_bei_20.append(abs(T[idx] - T_exakt_20))

# --- Visualisierung ---
fig, ax = plt.subplots()
ax.loglog(schrittweiten, fehler_bei_20, marker='o')
ax.set_xlabel('Schrittweite $\\Delta t$ in min')
ax.set_ylabel('Absoluter Fehler in °C')
ax.set_title('Globaler Fehler bei $t = 20$ min')
ax.grid(True, which='both')
plt.tight_layout()
plt.show()

Der Plot des globalen Fehlers zeigt eine typische **Konvergenzgerade** im
doppelt logarithmischen Maßstab: Halbieren wir die Schrittweite, halbiert sich
auch der Fehler ungefähr. Das bedeutet einen globalen Fehler von $O(\Delta t)$,
das Euler-Verfahren hat Konvergenzordnung 1. Zum Vergleich: Die Trapezregel aus
Kapitel 9 hatte Ordnung $O(\Delta t^2)$. In Abschnitt 10.3 lernen wir Verfahren
kennen, die Ordnung 4 oder höher erreichen.

### Mini-Übung 3

1. Lassen Sie die Fehler bei 20 min ausgeben. Um welchen Faktor ändert sich der Fehler,
   wenn die Schrittweite von $\Delta t = 2.0$ auf $\Delta t = 1.0$ min
   halbiert wird? Stimmt das mit der erwarteten Konvergenzordnung $O(\Delta t)$
   überein?
2. Schätzen Sie ohne Code: Welche Schrittweite brauchen Sie ungefähr, damit
   der Fehler bei $t = 20$ min unter $0.001\,°C$ liegt? Welchen Rechenaufwand
   (Anzahl Schritte) hätte das?
3. Das Euler-Verfahren hat Konvergenzordnung 1. Was bedeutet das konkret für
   die Effizienz: Wie viel mehr Rechenaufwand braucht man, um den Fehler um
   den Faktor 1000 zu verringern?

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Eine gewöhnliche Differentialgleichung 1. Ordnung verknüpft eine unbekannte
Funktion mit ihrer Ableitung: $\dot{T} = f(t, T)$. Das explizite Euler-Verfahren
löst sie schrittweise: vom aktuellen Zustand $T_i$ wird die Steigung $f(t_i,
T_i)$ berechnet und ein Schritt der Länge $\Delta t$ ausgeführt. Das Verfahren
ist einfach zu implementieren, hat aber Konvergenzordnung 1: für eine zehnfache
Genauigkeit braucht man zehnmal mehr Schritte.

Im nächsten Abschnitt lernen wir `scipy.integrate.solve_ivp` kennen, das mit
einem Runge-Kutta-Verfahren arbeitet und bei gleichem Rechenaufwand eine
vielfach höhere Genauigkeit erreicht.